# Lecture 5 · Notebook 3 — Base vs. instruction-tuned: prompting a small LLM
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture05/lecture05_3_base_vs_instruct.ipynb)

**Companion to [Chapter 3](https://nlp-course-cuhksz.github.io/lecture05.html#p3-s34) and [Chapter 8 (slides 106–119)](https://nlp-course-cuhksz.github.io/lecture05.html#p8-s106) of the video lecture.**
A pretrained *base* model continues text; an *instruction-tuned* (SFT) model answers. We use a pair of small open models from the same
family, **SmolLM2-135M** (base) and **SmolLM2-135M-Instruct**, and:

1. ask both the moon-landing question from slide 106,
2. compare zero-shot and few-shot prompting (in-context learning, slide 34) by scoring next-token probabilities,
3. look at the chat template — what SFT data really looks like to the model, and
4. build the "loss on the output only" mask of slide 119 and measure what SFT changed.

Runs on CPU in about five minutes (two downloads of about 270 MB each). 135M parameters is tiny — far below the sizes where the
lecture says abilities emerge — so watch for both what works and what doesn't.

In [ ]:
!pip -q install transformers   # already installed on Colab; needed elsewhere

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
BASE, INSTRUCT = "HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(INSTRUCT)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to(device).eval()
inst = AutoModelForCausalLM.from_pretrained(INSTRUCT, dtype=torch.float32).to(device).eval()
print(f"{sum(p.numel() for p in base.parameters()) / 1e6:.0f}M parameters each, on {device}")

## 1 · Language modeling ≠ assisting users (slide 106)
The base model gets the raw prompt and continues it. The instruct model gets the same words wrapped in its **chat template**.

In [ ]:
@torch.no_grad()
def complete(model, prompt, max_new_tokens=60):
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.2, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def chat(model, question, max_new_tokens=80):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True)
    return complete(model, prompt, max_new_tokens)

q = "Explain the moon landing to a 6 year old in a few sentences."
print("BASE     :", complete(base, q))
print()
print("INSTRUCT :", chat(inst, q))

The base model continues the text in whatever shape such text usually takes on the web — often a numbered list that drifts off topic.
Modern base models have read a lot of question-and-answer text, so they sometimes *look* as if they answer, but nothing makes them stay
on the request or stop. The instruct model treats the line as a request. (Its answer may still be wrong: 135M parameters know little.)
Try a few more:

In [ ]:
for q in ["Give me three tips for learning Python.", "Translate to French: I love natural language processing."]:
    print("Q:", q)
    print("  BASE     :", complete(base, q, 40).replace("\n", " / "))
    print("  INSTRUCT :", chat(inst, q, 60).replace("\n", " / "))
    print()

The instruct model follows the *format* of both requests, but its French is poor: SFT unlocks behaviour the base model already has
(slide 50); it cannot add much ability to a 135M model.

## 2 · Zero-shot vs. few-shot prompting (slides 34–35)
We classify short movie reviews with the **base** model, no training at all. Instead of parsing generated text, we compare the probability
of the next token being " positive" or " negative" — a clean way to read a model's choice.

In [ ]:
reviews = [
    ("A wonderful film with brilliant acting and a moving story.", "positive"),
    ("I loved every minute of it, the music is fantastic.", "positive"),
    ("One of the best movies I have seen this year.", "positive"),
    ("The cast is charming and the ending made me smile.", "positive"),
    ("Beautifully shot, funny and surprisingly touching.", "positive"),
    ("A delightful surprise, I would happily watch it again.", "positive"),
    ("Smart, warm and full of great performances.", "positive"),
    ("An instant classic that the whole family will enjoy.", "positive"),
    ("A boring mess with terrible dialogue.", "negative"),
    ("I walked out halfway through, it was that bad.", "negative"),
    ("The plot makes no sense and the acting is wooden.", "negative"),
    ("A complete waste of time and money.", "negative"),
    ("Dull, predictable and far too long.", "negative"),
    ("The jokes fall flat and the characters are annoying.", "negative"),
    ("I regret buying a ticket for this awful film.", "negative"),
    ("Poorly written, badly directed, and painfully slow.", "negative"),
]
demos = [("This movie sucks.", "negative"), ("I love this movie.", "positive"),
         ("What a fantastic, heartfelt story.", "positive"), ("Terrible acting and a stupid plot.", "negative")]
label_ids = {lab: tok(" " + lab, add_special_tokens=False)["input_ids"][0] for lab in ["positive", "negative"]}

@torch.no_grad()
def classify(model, review, shots):
    prompt = "".join(f"Review: {r}\nSentiment: {l}\n\n" for r, l in shots) + f"Review: {review}\nSentiment:"
    logits = model(**tok(prompt, return_tensors="pt").to(device)).logits[0, -1]
    lp = F.log_softmax(logits, -1)
    return max(label_ids, key=lambda lab: lp[label_ids[lab]].item()), {k: lp[v].exp().item() for k, v in label_ids.items()}

for k in [0, 1, 2, 4]:
    preds = [classify(base, r, demos[:k])[0] for r, _ in reviews]
    acc = sum(p == y for p, (_, y) in zip(preds, reviews)) / len(reviews)
    print(f"{k}-shot: accuracy {acc:.0%}   predicted positive {preds.count('positive'):2d} / negative {preds.count('negative'):2d}")

Read the rows carefully. When we ran this notebook, zero-shot already got 88% (these reviews are easy), one or two demonstrations in the
context — no weight update — fixed the remaining errors, but with **four** demonstrations the model labelled *every* review "negative",
the label of the last demonstration. Small models are known to be swayed by the order and the labels of the demonstrations
(see *Calibrate Before Use*, Zhao et al., 2021). In-context learning is real but fragile at this size; the lecture's point is that it
becomes reliable only in much larger models (exercise 2).

## 3 · What SFT data looks like: the chat template (slide 107)
Instruction tuning trains on conversations rendered into a fixed text format with special tokens. This is the exact string the instruct
model was trained on:

In [ ]:
conversation = [
    {"role": "user", "content": "Classify into animals, plants and minerals: oak tree, copper ore, elephant."},
    {"role": "assistant", "content": "Oak tree: plant. Copper ore: mineral. Elephant: animal."},
    {"role": "user", "content": "Add one more of each."},
    {"role": "assistant", "content": "Rose: plant. Quartz: mineral. Tiger: animal."},
]
print(tok.apply_chat_template(conversation, tokenize=False))

## 4 · Loss on the output only (slide 119)
During SFT the whole conversation is fed to the model, but the loss is computed only on the **assistant** tokens; system and user tokens
get the label `-100` (ignored by PyTorch's cross-entropy). We find the assistant spans by rendering the conversation turn by turn.

In [ ]:
def chat_ids(conv, **kw):
    out = tok.apply_chat_template(conv, tokenize=True, **kw)
    return list(out["input_ids"] if hasattr(out, "keys") else out)   # newer transformers versions may return a dict

def build_labels(conv):
    ids = chat_ids(conv)
    labels = [-100] * len(ids)
    for i, msg in enumerate(conv):
        if msg["role"] != "assistant":
            continue
        start = len(chat_ids(conv[:i], add_generation_prompt=True))
        end = len(chat_ids(conv[:i + 1]))
        assert ids[:end] == chat_ids(conv[:i + 1])   # the template is prefix-consistent
        labels[start:end] = ids[start:end]
    return torch.tensor([ids]), torch.tensor([labels])

input_ids, labels = build_labels(conversation)
trained = (labels[0] != -100).sum().item()
print(f"{input_ids.shape[1]} tokens in the conversation, loss on {trained} of them\n")
print("".join(f"[{tok.decode([t])}]" if l != -100 else tok.decode([t]) for t, l in zip(input_ids[0].tolist(), labels[0].tolist())))

Tokens in `[brackets]` are the ones the model learns to produce. Now compare the two models' loss on exactly those tokens:

In [ ]:
@torch.no_grad()
def lm_loss(model, input_ids, labels):
    logits = model(input_ids.to(device)).logits[:, :-1]
    return F.cross_entropy(logits.flatten(0, 1), labels[:, 1:].flatten().to(device), ignore_index=-100).item()

all_labels = input_ids.clone()
for name, m in [("base", base), ("instruct", inst)]:
    print(f"{name:9s} loss on assistant tokens: {lm_loss(m, input_ids, labels):.3f}   on all tokens: {lm_loss(m, input_ids, all_labels):.3f}")

The instruct model is much less surprised by the assistant turns — and by the template's special tokens, which the base model never saw
in the role SFT gave them. Same architecture, same pretraining; a comparatively small amount of instruction data (Chapter 8's
*superficial alignment* hypothesis) changed which continuations the model considers likely.

## Exercises
1. Ask both models a factual question ("What is the capital of Australia?") and a harmful one. Does SFT add knowledge, or mostly
   change format and behaviour (slides 111–113)?
2. In section 2, swap the label words to " good" / " bad", or reorder the demonstrations. How stable is in-context learning at 135M parameters?
   Replace `BASE` with `HuggingFaceTB/SmolLM2-360M` (about 720 MB) — does a bigger model do better?
3. Write a multi-turn conversation with a system message and check with `build_labels` that only assistant turns get a loss. What would go
   wrong if the user turns were trained on too (slide 119)?

**Back to the [video lecture]({SITE})** · To fine-tune a model yourself, continue with
[Tutorial 2: Training Your Own LLMs](https://nlp-course-cuhksz.github.io/tutorial2.html).